# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from collections import defaultdict
from itertools import product

import pandas as pd
import yaml

from 01_evaluate__prompts import get_filepath
from src.prompts import *
from src.sets import *

In [ ]:
ANSWERS_POSITIVE = ['true', 'yes']
ANSWERS_NEGATIVE = ['false', 'no']

In [ ]:
setting = 'natural_3_4'

models = [
    {'path': 'meta-llama/Llama-3.2-3B-Instruct', 'name': 'llama_3b'},
    {'path': 'meta-llama/Llama-3.1-8B-Instruct', 'name': 'llama_8b'},
    {'path': 'meta-llama/Llama-3.1-70B-Instruct', 'name': 'llama_70b_3.1'},
    {'path': 'meta-llama/Llama-3.3-70B-Instruct', 'name': 'llama_70b_3.3'},
    {'path': 'microsoft/Phi-3.5-MoE-instruct', 'name': 'phi_moe'},
    {'path': 'Qwen/Qwen2.5-32B-Instruct', 'name': 'qwen_32b'},
    {'path': 'mistralai/Mistral-Small-24B-Instruct-2501', 'name': 'mistral_24b'}
]

fp_sets = './data/config/sets.yaml'
fp_prompts = f'./data/config/prompts_{setting}.yaml'
results_dir = f'./data/results/results_{setting}'

# Load data

In [ ]:
with open(fp_sets, mode='rb') as f:
    set_data = yaml.safe_load(f)

set_names = list(set_data.keys())

In [ ]:
with open(fp_prompts, mode='rb') as f:
    prompt_data = yaml.safe_load(f)

prefix = prompt_data['prefix']

templates = prompt_data['templates']

features = prompt_data['features']
feature_grid = get_feature_grid(features)
feature_names = get_feature_names(features, templates)

In [ ]:
permutations = generate_setinclusion_permutations([1, 2, 3, 4], intruder=0)
permutations = [{'e0': p['element']} | {f'e{i}': e for i, e in enumerate(p['elements'], start=1)} for p in permutations]

# Create DataFrame

In [ ]:
results = defaultdict(dict)

for model in models:

    for template, feature in product(templates, feature_grid):

        fids = get_feature_ids(feature, template)

        for set_name in set_names:

            fp = get_filepath(results_dir, model['path'], template, set_name, feature)
            with open(fp, encoding='utf-8') as f:
                data = json.load(f)
            
            for i, d in enumerate(data):

                # This condition is for thinking qwen only to drop the yapp
                if '</think>' in d['answer']:
                    answer = d['answer'].split('</think>')[-1]

                answer = create_set_of_words(d['answer'])
                has_positive = any((True for w in answer if w in ANSWERS_POSITIVE))
                has_negative = any((True for w in answer if w in ANSWERS_NEGATIVE))

                y_true = d['element'] in d['elements']

                if (has_positive and has_negative) or (not has_positive and not has_negative):
                    y_pred = 'x' if y_true else 'x'
                else:
                    if y_true and has_positive:
                        y_pred = 'tp'
                    elif not y_true and has_negative:
                        y_pred = 'tn'
                    elif not y_true and has_positive:
                        y_pred = 'fp'
                    elif y_true and has_negative:
                        y_pred = 'fn'
            
                key = tuple(list(fids.items()) + list(permutations[i].items()))
                results[key][f'{model["name"]}_{set_name}'] = y_pred

In [ ]:
result_list = list()
for r in results:
    result_list.append(dict(r) | results[r])

In [ ]:
df = pd.DataFrame.from_dict(result_list)
df = df.replace(get_feature_id_mapping(features, templates))
df

In [ ]:
df.to_csv(f'data/tables/{setting}_raw.csv', sep=';')